## 1. Import thư viện

In [1]:
import os
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

## 2. Khai báo đường dẫn dữ liệu

In [2]:
train_path = "../data/multilexnorm2026_vi_train.csv"
val_path   = "../data/multilexnorm2026_vi_validation.csv"
test_path  = "../data/multilexnorm2026_vi_test.csv"

for path in [train_path, val_path, test_path]:
    print(path, "->", "FOUND" if os.path.exists(path) else "NOT FOUND")

../data/multilexnorm2026_vi_train.csv -> FOUND
../data/multilexnorm2026_vi_validation.csv -> FOUND
../data/multilexnorm2026_vi_test.csv -> FOUND


## 3. Đọc dữ liệu

In [3]:
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print("Train shape     :", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape      :", test_df.shape)

Train shape     : (8371, 3)
Validation shape: (1050, 3)
Test shape      : (522, 3)


## 4. Xem vài dòng đầu

In [4]:
train_df.head()

,raw,norm,lang
0,['thích' 'anh' 'cá' 'mập' 'k'],['thích' 'anh' 'cá' 'mập' 'không'],vi
1,['cứ' 'ngây' 'thơ' 'thế' 'thoai' ':))'],['cứ' 'ngây' 'thơ' 'thế' 'thôi' ':))'],vi
2,['bà' 'Nghê' 'xinh' 'vậy' 'mà' 't' 'thấy' 'k' ...,['bà' 'Nghê' 'xinh' 'vậy' 'mà' 'tôi' 'thấy' 'k...,vi
3,['Ê' 'k' 'khóc' 'được' 'làm' 'thế' 'nào' 'má' ...,['Ê' 'không' 'khóc' 'được' 'làm' 'thế' 'nào' '...,vi
4,['Có' 'biến' 'gì' 'hong' 'dẫy' ':))'],['Có' 'biến' 'gì' 'không' 'vậy' ':))'],vi


## 5. Parse cột `raw` và `norm` thành list token

Đổi dữ liệu dạng chuỗi về python list

In [5]:
def parse_token_list(text):
    text = str(text)
    tokens = re.findall(r"'([^']*)'", text)
    return tokens

In [6]:
for df in [train_df, val_df, test_df]:
    df["raw_tokens"] = df["raw"].apply(parse_token_list)
    df["norm_tokens"] = df["norm"].apply(parse_token_list)

## 6. Xem thử một vài ví dụ sau khi parse

In [7]:
for i in range(3):
    print(f"--- Example {i} ---")
    print("RAW :", train_df.loc[i, "raw_tokens"])
    print("NORM:", train_df.loc[i, "norm_tokens"])
    print()

--- Example 0 ---
RAW : ['thích', 'anh', 'cá', 'mập', 'k']
NORM: ['thích', 'anh', 'cá', 'mập', 'không']

--- Example 1 ---
RAW : ['cứ', 'ngây', 'thơ', 'thế', 'thoai', ':))']
NORM: ['cứ', 'ngây', 'thơ', 'thế', 'thôi', ':))']

--- Example 2 ---
RAW : ['bà', 'Nghê', 'xinh', 'vậy', 'mà', 't', 'thấy', 'k', 'bằng', 'bà', 'ChiPu', 'luôn', 'chời']
NORM: ['bà', 'Nghê', 'xinh', 'vậy', 'mà', 'tôi', 'thấy', 'không', 'bằng', 'bà', 'ChiPu', 'luôn', 'trời']



## 7. Kiểm tra cột dữ liệu

In [8]:
print("Train columns     :", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())
print("Test columns      :", test_df.columns.tolist())

Train columns     : ['raw', 'norm', 'lang', 'raw_tokens', 'norm_tokens']
Validation columns: ['raw', 'norm', 'lang', 'raw_tokens', 'norm_tokens']
Test columns      : ['raw', 'norm', 'lang', 'raw_tokens', 'norm_tokens']


## 8. Thống kê số câu

In [9]:
print("Number of train sentences     :", len(train_df))
print("Number of validation sentences:", len(val_df))
print("Number of test sentences      :", len(test_df))

Number of train sentences     : 8371
Number of validation sentences: 1050
Number of test sentences      : 522


## 9. Thống kê số token

In [10]:
def count_total_tokens(token_series):
    return sum(len(tokens) for tokens in token_series)

train_total_tokens = count_total_tokens(train_df["raw_tokens"])
val_total_tokens   = count_total_tokens(val_df["raw_tokens"])
test_total_tokens  = count_total_tokens(test_df["raw_tokens"])

print("Train total tokens     :", train_total_tokens)
print("Validation total tokens:", val_total_tokens)
print("Test total tokens      :", test_total_tokens)

Train total tokens     : 101773
Validation total tokens: 13649
Test total tokens      : 6362


## 10. Độ dài câu trung bình

In [11]:
train_df["sent_len"] = train_df["raw_tokens"].apply(len)
val_df["sent_len"]   = val_df["raw_tokens"].apply(len)
test_df["sent_len"]  = test_df["raw_tokens"].apply(len)

print("Average train sentence length     :", round(train_df["sent_len"].mean(), 2))
print("Average validation sentence length:", round(val_df["sent_len"].mean(), 2))
print("Average test sentence length      :", round(test_df["sent_len"].mean(), 2))

Average train sentence length     : 12.16
Average validation sentence length: 13.0
Average test sentence length      : 12.19


## 11. Kiểm tra `raw` và `norm` có cùng số token không

Bước này quan trọng vì baseline lookup ở giai đoạn đầu giả định mỗi `raw token` khớp với đúng một `norm token`.

In [12]:
def count_length_mismatch(df):
    mismatch = 0
    for raw_tokens, norm_tokens in zip(df["raw_tokens"], df["norm_tokens"]):
        if len(raw_tokens) != len(norm_tokens):
            mismatch += 1
    return mismatch

train_mismatch = count_length_mismatch(train_df)
val_mismatch   = count_length_mismatch(val_df)

print("Train length mismatches     :", train_mismatch)
print("Validation length mismatches:", val_mismatch)

Train length mismatches     : 8
Validation length mismatches: 0


## 12. Tỷ lệ token cần chuẩn hóa

Một token được xem là **cần chuẩn hóa** nếu `raw != norm`.

In [13]:
def normalization_stats(df):
    total = 0
    changed = 0

    for raw_tokens, norm_tokens in zip(df["raw_tokens"], df["norm_tokens"]):
        if len(raw_tokens) != len(norm_tokens):
            continue

        for raw_tok, norm_tok in zip(raw_tokens, norm_tokens):
            total += 1
            if raw_tok != norm_tok:
                changed += 1

    unchanged = total - changed
    ratio = changed / total if total > 0 else 0
    return total, changed, unchanged, ratio

train_total, train_changed, train_unchanged, train_ratio = normalization_stats(train_df)
val_total, val_changed, val_unchanged, val_ratio = normalization_stats(val_df)

print("TRAIN")
print("Total tokens     :", train_total)
print("Changed tokens   :", train_changed)
print("Unchanged tokens :", train_unchanged)
print("Changed ratio    :", round(train_ratio * 100, 2), "%")

print("\nVALIDATION")
print("Total tokens     :", val_total)
print("Changed tokens   :", val_changed)
print("Unchanged tokens :", val_unchanged)
print("Changed ratio    :", round(val_ratio * 100, 2), "%")

TRAIN
Total tokens     : 101665
Changed tokens   : 16256
Unchanged tokens : 85409
Changed ratio    : 15.99 %

VALIDATION
Total tokens     : 13649
Changed tokens   : 2135
Unchanged tokens : 11514
Changed ratio    : 15.64 %


## 13. Token phổ biến trong `raw`

In [14]:
raw_counter = Counter()

for tokens in train_df["raw_tokens"]:
    raw_counter.update(tokens)

raw_counter.most_common(20)

[('có', 1404),
 ('là', 1329),
 ('t', 1106),
 ('mà', 1041),
 ('thì', 1021),
 ('ko', 926),
 ('k', 915),
 ('đi', 874),
 ('cho', 805),
 ('mình', 804),
 ('ăn', 742),
 ('cũng', 740),
 ('này', 736),
 ('cái', 613),
 ('đc', 536),
 ('của', 509),
 ('bạn', 505),
 ('làm', 500),
 ('như', 497),
 ('thấy', 493)]

## 14. Token phổ biến trong `norm`

In [15]:
norm_counter = Counter()

for tokens in train_df["norm_tokens"]:
    norm_counter.update(tokens)

norm_counter.most_common(20)

[('không', 2730),
 ('có', 1439),
 ('là', 1345),
 ('tôi', 1222),
 ('được', 1092),
 ('mà', 1071),
 ('thì', 1031),
 ('đi', 966),
 ('mình', 964),
 ('cũng', 906),
 ('rồi', 898),
 ('cho', 809),
 ('này', 790),
 ('ăn', 759),
 ('em', 732),
 ('bạn', 710),
 ('vậy', 709),
 ('người', 627),
 ('cái', 621),
 ('làm', 572)]

## 15. Các cặp `raw -> norm` phổ biến nhất

In [16]:
pair_counter = Counter()

for raw_tokens, norm_tokens in zip(train_df["raw_tokens"], train_df["norm_tokens"]):
    if len(raw_tokens) != len(norm_tokens):
        continue

    for raw_tok, norm_tok in zip(raw_tokens, norm_tokens):
        pair_counter[(raw_tok, norm_tok)] += 1

pair_counter.most_common(30)

[(('có', 'có'), 1402),
 (('là', 'là'), 1328),
 (('mà', 'mà'), 1040),
 (('thì', 'thì'), 1019),
 (('t', 'tôi'), 933),
 (('ko', 'không'), 917),
 (('k', 'không'), 911),
 (('đi', 'đi'), 873),
 (('cho', 'cho'), 805),
 (('mình', 'mình'), 804),
 (('ăn', 'ăn'), 742),
 (('cũng', 'cũng'), 740),
 (('này', 'này'), 736),
 (('cái', 'cái'), 612),
 (('đc', 'được'), 527),
 (('của', 'của'), 509),
 (('bạn', 'bạn'), 504),
 (('làm', 'làm'), 498),
 (('như', 'như'), 497),
 (('thấy', 'thấy'), 493),
 (('còn', 'còn'), 491),
 (('con', 'con'), 477),
 (('ra', 'ra'), 475),
 (('phải', 'phải'), 458),
 (('chứ', 'chứ'), 424),
 (('nó', 'nó'), 421),
 (('1', '1'), 419),
 (('nào', 'nào'), 415),
 (('lại', 'lại'), 395),
 (('để', 'để'), 391)]

## 16. Chỉ lấy các cặp có thay đổi thật sự

Tức là chỉ lấy các cặp mà `raw != norm`.

In [17]:
changed_pair_counter = Counter()

for (raw_tok, norm_tok), count in pair_counter.items():
    if raw_tok != norm_tok:
        changed_pair_counter[(raw_tok, norm_tok)] = count

changed_pair_counter.most_common(30)

[(('t', 'tôi'), 933),
 (('ko', 'không'), 917),
 (('k', 'không'), 911),
 (('đc', 'được'), 527),
 (('e', 'em'), 333),
 (('r', 'rồi'), 314),
 (('ng', 'người'), 266),
 (('m', 'mày'), 232),
 (('z', 'vậy'), 209),
 (('dc', 'được'), 174),
 (('a', 'anh'), 172),
 (('vs', 'với'), 172),
 (('ngta', 'người ta'), 159),
 (('ck', 'chồng'), 155),
 (('t', 'tao'), 152),
 (('b', 'bạn'), 151),
 (('iu', 'yêu'), 150),
 (('ny', 'người yêu'), 136),
 (('kh', 'không'), 135),
 (('khum', 'không'), 125),
 (('lun', 'luôn'), 116),
 (('T', 'Tôi'), 113),
 (('j', 'gì'), 109),
 (('v', 'vậy'), 93),
 (('cx', 'cũng'), 87),
 (('dị', 'vậy'), 81),
 (('h', 'giờ'), 80),
 (('trc', 'trước'), 77),
 (('mn', 'mọi người'), 76),
 (('tr', 'trời'), 75)]

## 17. Kiểm tra một vài token quen thuộc

Cell này giúp mình nhìn nhanh xem một token noisy thường được chuẩn hóa thành gì.

In [18]:
for token in ["ko", "k", "đc", "dc", "e", "r", "khum", "ngta", "t", "z"]:
    candidates = []
    for (raw_tok, norm_tok), count in pair_counter.items():
        if raw_tok == token:
            candidates.append((norm_tok, count))
    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)
    print(f"{token} -> {candidates[:10]}")

ko -> [('không', 917), ('ko', 8), ('khôngo', 1)]
k -> [('không', 911), ('k', 4)]
đc -> [('được', 527), ('đc', 8), ('đượ', 1)]
dc -> [('được', 174), ('dc', 2)]
e -> [('em', 333), ('e', 6), ('en', 2)]
r -> [('rồi', 314), ('rượu', 1), ('rôi', 1)]
khum -> [('không', 125)]
ngta -> [('người ta', 159), ('ngta', 4), ('người', 1)]
t -> [('tôi', 933), ('tao', 152), ('tớ', 8), ('t', 5), ('tui', 4), ('thằng', 1), ('tồi', 1), ('ta', 1), ('tới', 1)]
z -> [('vậy', 209), ('z', 2)]


## 18. Tìm token mơ hồ

Token mơ hồ là một `raw token` có thể map tới **nhiều hơn một** `norm token`.

In [19]:
raw_to_norms = defaultdict(Counter)

for raw_tokens, norm_tokens in zip(train_df["raw_tokens"], train_df["norm_tokens"]):
    if len(raw_tokens) != len(norm_tokens):
        continue

    for raw_tok, norm_tok in zip(raw_tokens, norm_tokens):
        raw_to_norms[raw_tok][norm_tok] += 1

ambiguous_tokens = []
for raw_tok, counter in raw_to_norms.items():
    if len(counter) > 1:
        ambiguous_tokens.append((raw_tok, len(counter), counter.most_common(5)))

ambiguous_tokens = sorted(ambiguous_tokens, key=lambda x: x[1], reverse=True)

for item in ambiguous_tokens[:20]:
    print(item)

('t', 9, [('tôi', 933), ('tao', 152), ('tớ', 8), ('t', 5), ('tui', 4)])
('n', 9, [('nó', 27), ('n', 5), ('nhiều', 3), ('nhưng', 1), ('người', 1)])
('m', 7, [('mày', 232), ('mình', 27), ('m', 17), ('má', 1), ('mẹ', 1)])
('đk', 7, [('được', 10), ('đúng không', 2), ('đk', 1), ('đăng kí', 1), ('được không', 1)])
('xí', 6, [('xí', 3), ('tí', 1), ('ít', 1), ('xấu', 1), ('sĩ', 1)])
('tr', 5, [('trời', 75), ('triệu', 13), ('tr', 2), ('trai', 1), ('truyện', 1)])
('đ', 5, [('đéo', 41), ('điểm', 2), ('đồng', 2), ('đếch', 1), ('đang', 1)])
('ch', 5, [('chưa', 9), ('chuyện', 3), ('chó', 2), ('ch', 1), ('chồng', 1)])
('ah', 5, [('ạ', 8), ('à', 5), ('anh', 5), ('ah', 1), ('á', 1)])
('tk', 5, [('thằng', 8), ('tài khoản', 7), ('thế', 1), ('tiết kiệm', 1), ('tk', 1)])
('hong', 4, [('không', 70), ('hông', 8), ('hong', 2), ('hóng', 1)])
('ck', 4, [('chồng', 155), ('chông', 1), ('chuyển khoản', 1), ('không', 1)])
('b', 4, [('bạn', 151), ('biết', 2), ('b', 1), ('bố', 1)])
('hè', 4, [('hè', 14), ('nè', 2), (

In [20]:
matches = []

for idx, row in train_df.iterrows():
    raw_tokens = row["raw_tokens"]
    norm_tokens = row["norm_tokens"]
    
    if len(raw_tokens) != len(norm_tokens):
        continue
        
    for raw_tok, norm_tok in zip(raw_tokens, norm_tokens):
        if raw_tok == "t" and norm_tok == "t":
            matches.append((idx, raw_tokens, norm_tokens))
            break

print("Number of sentences containing t -> t:", len(matches))

for i, item in enumerate(matches[:10]):
    sent_idx, raw_sent, norm_sent = item
    print(f"\nSentence index: {sent_idx}")
    print("RAW :", raw_sent)
    print("NORM:", norm_sent)

Number of sentences containing t -> t: 5

Sentence index: 365
RAW : ['t', 'vca', 'mạnh', 'mèo', 'nhuộm', 'sập', 'còn', 'kh', 'xơ', 'bằng', 'con', 'ty']
NORM: ['t', 'vca', 'mạnh', 'mèo', 'nhuộm', 'sập', 'còn', 'không', 'xơ', 'bằng', 'con', 'ty']

Sentence index: 851
RAW : ['lớp', 't', 'thi', 'tin', 'cuối', 'kì', '40p,', '5p', 'thầy', 'chấm', 'bài', '+', 'vô', 'điểm', 'lớp', 't', '46', 'đứa,', 'thầy', 'quá', 'đỉnh', '=))))']
NORM: ['lớp', 'tôi', 'thi', 'tin', 'cuối', 'kì', '40p,', '5p', 'thầy', 'chấm', 'bài', '+', 'vô', 'điểm', 'lớp', 't', '46', 'đứa,', 'thầy', 'quá', 'đỉnh', '=))))']

Sentence index: 5086
RAW : ['Ề', 't', 'cũng', 'thích', 'con', 'này', 'mà', 'cứ', 'tưởng', 'k', 'đc', 'nuôi', 'chớ']
NORM: ['Ê', 't', 'cũng', 'thích', 'con', 'này', 'mà', 'cứ', 'tưởng', 'không', 'được', 'nuôi', 'chứ']

Sentence index: 5455
RAW : ['Có', 'tháng', 'ck', 't', 'còn', 'dưới', '10', 'chẹo,', 'và', 'sắp', 'sửa', 'nợ', 'hơi', 'bị', 'nhiều']
NORM: ['Có', 'tháng', 'chồng', 't', 'còn', 'dưới', '10', 't

## 19. Xây vocab của train / validation / test

In [21]:
train_vocab = set()
val_vocab = set()
test_vocab = set()

for tokens in train_df["raw_tokens"]:
    train_vocab.update(tokens)

for tokens in val_df["raw_tokens"]:
    val_vocab.update(tokens)

for tokens in test_df["raw_tokens"]:
    test_vocab.update(tokens)

val_oov = val_vocab - train_vocab
test_oov = test_vocab - train_vocab

print("Train vocab size            :", len(train_vocab))
print("Validation vocab size       :", len(val_vocab))
print("Test vocab size             :", len(test_vocab))
print("Validation OOV size vs train:", len(val_oov))
print("Test OOV size vs train      :", len(test_oov))

Train vocab size            : 9768
Validation vocab size       : 3272
Test vocab size             : 2059
Validation OOV size vs train: 689
Test OOV size vs train      : 355


## 20. Tỷ lệ OOV theo token


In [23]:
def oov_token_stats(df, train_vocab):
    total = 0
    oov = 0
    for tokens in df["raw_tokens"]:
        for tok in tokens:
            total += 1
            if tok not in train_vocab:
                oov += 1
    ratio = oov / total if total > 0 else 0
    return total, oov, ratio

val_total_tok, val_oov_tok, val_oov_ratio = oov_token_stats(val_df, train_vocab)
test_total_tok, test_oov_tok, test_oov_ratio = oov_token_stats(test_df, train_vocab)

print("Validation OOV tokens:", val_oov_tok, "/", val_total_tok, "=", round(val_oov_ratio * 100, 2), "%")
print("Test OOV tokens      :", test_oov_tok, "/", test_total_tok, "=", round(test_oov_ratio * 100, 2), "%")

Validation OOV tokens: 727 / 13649 = 5.33 %
Test OOV tokens      : 373 / 6362 = 5.86 %


## 21. In ra một số token OOV để xem nhanh

In [22]:
print("Some validation OOV tokens:")
print(sorted(list(val_oov))[:100])

Some validation OOV tokens:
['"1', '"bả', '"bẻ', '"bới"', '"cái', '"là', '"mệt', '"thì', '"trần', '"tình', '"Ủa', '"ủa,', '#', '(này', '(╭￣3￣)╭♡', ')" ', '***', ',chắc', ',cứ', ',dc', ',không', ',mẹ', ',vk', '.e', '1,2,', '10,', '100m', '10?', '10triệu/tháng', '10đ,', '15k', '1930', '1m68', '2022', '20m', '26t', '2chục', '30-40', '30km', '31', '44-45,', '46t', '4t', '5,1tr', '50-60t', '500Tr', '50k/1', '550k', '5:20AM', '5m', '650k/kg', '67kg', '82kg', '900', '915', ':))))))))))))))', ':,))))', ':vv', '=="', '=]', '=]]]]]]', '=_=-', 'Atlas', 'Bazaar', 'Beo', 'Bidv,', 'Biên', 'Bình', 'Bnhiu', 'Bucky', 'BÂY', 'Bàn', 'Bí', 'Bắp', 'BỊ', 'CAND', 'CHÊ', 'CV', 'Chac', 'Chai', 'ChatGPT', 'ChauMinh', 'Chiến', 'Chiều', 'Chuyến', 'Chòe', 'Cienco', 'Ckac', 'Csong', 'Cám', 'Cậy', 'Dìa', 'Duocc', 'Dơ', 'Dụ', 'Endgame', 'Fury', 'GIỜ', 'Giáp', 'Giọng']


## 22. Tạo lookup table sơ bộ từ train

In [24]:
lookup_table = {}

for raw_tok, counter in raw_to_norms.items():
    lookup_table[raw_tok] = counter.most_common(1)[0][0]

print("Lookup table size:", len(lookup_table))

Lookup table size: 9765


In [25]:
for token in ["ko", "k", "đc", "dc", "e", "t", "ngta", "khum"]:
    print(token, "->", lookup_table.get(token, token))

ko -> không
k -> không
đc -> được
dc -> được
e -> em
t -> tôi
ngta -> người ta
khum -> không


## 23. Lưu một vài thống kê ra file

In [26]:
os.makedirs("../outputs", exist_ok=True)

top_pairs_df = pd.DataFrame(
    [(raw, norm, count) for (raw, norm), count in changed_pair_counter.most_common(200)],
    columns=["raw", "norm", "count"]
)

save_path = "../outputs/top_changed_pairs.csv"
top_pairs_df.to_csv(save_path, index=False, encoding="utf-8-sig")
print("Saved:", save_path)

Saved: ../outputs/top_changed_pairs.csv


## 24. Kết luận nhanh

Sau notebook này, mình nên trả lời được các câu hỏi:

- Dữ liệu có bao nhiêu câu và bao nhiêu token?
- Tỷ lệ token cần chuẩn hóa là bao nhiêu?
- Có nhiều mapping lặp lại không?
- Có token mơ hồ không?
- Validation/Test có nhiều OOV không?
- Hướng lookup table có tiềm năng không?

Nếu các câu trả lời đều rõ, bước tiếp theo rất hợp lý là làm notebook:

**`02_baseline_lookup.ipynb`**